In [1]:
from pymongo import MongoClient

client = MongoClient("127.0.0.1", 27017)
db = client["api_update"]
col = db["java_existent_api_update_instances"]
col.estimated_document_count()

58092

In [2]:
from tqdm import tqdm
import pandas as pd
from packaging.version import Version

tqdm.pandas()
commit_pairs_orig = []

for doc in tqdm(col.find({}), total=col.estimated_document_count()):
    commit = doc["commit"]
    version_before = doc["version_before"]

    version_after = doc["version_after"]
    for pair in doc["api_update_pairs"]:
        old_callee = pair["old_callee"]
        old_api = old_callee["full_name"]
        old_body = old_callee["body"]
        new_callee = pair["new_callee"]
        new_api = new_callee["full_name"]
        new_body = new_callee["body"]

        if (len(new_body) == 0) and (len(old_body) > 0):
            continue
        if (len(new_body) > 0) and (len(old_body) == 0):
            continue

        record = [
            doc["package"],
            version_before,
            version_after,
            old_api,
            new_api,
            commit,
        ]
        old_v = Version(version_before)
        new_v = Version(version_after)

        if old_v == new_v:
            continue

        # old_api -> new_api, up
        # new_api -> old_api, down
        elif old_v < new_v:
            if old_api < new_api:
                rule = [(old_api, new_api), "Up"]
            else:
                rule = [(new_api, old_api), "Down"]

        # old_api -> new_api, down
        # new_api -> old_api, up
        else:
            if old_api < new_api:
                rule = [(old_api, new_api), "Down"]
            else:
                rule = [(new_api, old_api), "Up"]
        commit_pairs_orig.append(record + rule)


commit_pairs_orig = (
    pd.DataFrame(
        commit_pairs_orig,
        columns=[
            "package",
            "version_before",
            "version_after",
            "old_api",
            "new_api",
            "commit",
            "rule",
            "direction",
        ],
    )
    .drop_duplicates()
    .dropna()
)

print(len(commit_pairs_orig), "commit pairs before filtering")
print(
    len(commit_pairs_orig[["package", "rule", "direction"]].drop_duplicates()),
    "rules before filtering",
)
print(
    len(
        commit_pairs_orig[
            ["package", "version_before", "version_after", "old_api", "new_api"]
        ].drop_duplicates()
    ),
    "pairs before filtering",
)

100%|██████████| 58092/58092 [00:12<00:00, 4545.48it/s]


40402 commit pairs before filtering
16080 rules before filtering
28697 pairs before filtering


In [3]:
def ratio_fun(row):
    if row["Down"] > row["Up"]:
        row["ratio"] = row["Down"] / row["Up"]
    else:
        row["ratio"] = row["Up"] / row["Down"]

    return row


def find_naive_error_rules():
    rule_df = (
        commit_pairs_orig.groupby(["package", "rule", "direction"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
        .rename_axis(None, axis=1)
    )
    candidate_error_rules = rule_df[(rule_df["Down"] > 0) & (rule_df["Up"] > 0)]
    print(f"{len(candidate_error_rules)} rules have both Up and Down direction")
    candidate_error_rules = candidate_error_rules.apply(ratio_fun, axis=1).sort_values(
        "ratio", ascending=False
    )
    data = []
    for row in candidate_error_rules.itertuples(index=False):
        if row.ratio < 5:
            data.append([row.package, row.rule, "Up"])
            data.append([row.package, row.rule, "Down"])
        else:
            if row.Up > row.Down:
                data.append([row.package, row.rule, "Down"])
            else:
                data.append([row.package, row.rule, "Up"])
    print(len(data), "error rules")
    error_rules = pd.DataFrame(data, columns=["package", "rule", "direction"])
    return error_rules


error_rules = find_naive_error_rules()

366 rules have both Up and Down direction
670 error rules


In [4]:
commit_pairs_filtered = (
    pd.merge(commit_pairs_orig, error_rules, indicator=True, how="left")
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
print(len(commit_pairs_filtered), "commit pairs after filtering")
print(
    len(commit_pairs_filtered[["package", "rule"]].drop_duplicates()),
    "rules after filtering",
)

38285 commit pairs after filtering
15410 rules after filtering


In [5]:
def canonical_pairs(row):
    version_before = row["version_before"]
    version_after = row["version_after"]
    package = row["package"]
    old_api = row["old_api"]
    new_api = row["new_api"]
    commit = row["commit"]
    if Version(version_before) > Version(version_after):
        return pd.Series(
            [package, version_after, version_before, new_api, old_api, commit],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api",
                "new_api",
                "commit",
            ],
        )
    else:
        return pd.Series(
            [package, version_before, version_after, old_api, new_api, commit],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api",
                "new_api",
                "commit",
            ],
        )


commit_pairs_full = commit_pairs_filtered.apply(canonical_pairs, axis=1)

In [6]:
rule_freq = (
    commit_pairs_full.groupby(["package", "old_api", "new_api"])["commit"]
    .nunique()
    .reset_index()
    .sort_values("commit", ascending=False, ignore_index=True)
)
num_packages = rule_freq["package"].nunique()
num_releases = len(
    pd.concat(
        [
            commit_pairs_full[["package", "old_version"]].rename(
                columns={"old_version": "version"}
            ),
            commit_pairs_full[["package", "new_version"]].rename(
                columns={"new_version": "version"}
            ),
        ]
    ).drop_duplicates()
)
num_rules = len(rule_freq)
num_commits = commit_pairs_full["commit"].nunique()
print(f"# Packages: {num_packages}")
print(f"# Releases: {num_releases}")
print(f"# pairs: {num_rules}")
print(f"# commits: {num_commits}")

# Packages: 2546
# Releases: 11990
# pairs: 15410
# commits: 15046


In [7]:
pair_gte10 = rule_freq[rule_freq["commit"] >= 10]
print(
    f"{len(pair_gte10)} rules in {pair_gte10['package'].nunique()} Java packages with freq >= 10"
)
pair_1to10 = rule_freq[(rule_freq["commit"] > 1) & (rule_freq["commit"] < 10)]
print(
    f"{len(pair_1to10)} rules in {pair_1to10['package'].nunique()} Java packages with freq > 1 and < 10"
)
pair_eq1 = rule_freq[rule_freq["commit"] == 1]
print(
    f"{len(pair_eq1)} rules in {pair_eq1['package'].nunique()} Java packages with freq = 1"
)

338 rules in 101 Java packages with freq >= 10
4117 rules in 892 Java packages with freq > 1 and < 10
10955 rules in 2189 Java packages with freq = 1


In [8]:
from utils import cal_sample_size

population_size = len(pair_1to10) + len(pair_eq1)
sample_size = cal_sample_size(population_size)
sample_size_1to10 = round(sample_size * len(pair_1to10) / population_size)
sample_size_eq1 = round(sample_size * len(pair_eq1) / population_size)
print(f"Sample size for all rules with freq < 10: {sample_size}")
print(f"Sample size for rules with freq > 1 and < 10: {sample_size_1to10}")
print(f"Sample size for rules with freq = 1: {sample_size_eq1}")
pair_gte10.to_excel("../benchmark/final/java_api_update_pairs_gte10.xlsx")
pair_1to10.sample(sample_size_1to10).to_excel(
    "../benchmark/final/java_api_update_pairs_1to10.xlsx", index=False
)
pair_eq1.sample(sample_size_eq1).to_excel(
    "../benchmark/final/java_api_update_pairs_eq1.xlsx", index=False
)

Sample size for all rules with freq < 10: 375
Sample size for rules with freq > 1 and < 10: 102
Sample size for rules with freq = 1: 273


In [9]:
pair_gte10_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_pairs_gte10-labelled.xlsx"
)
correct_rules_gte10 = pair_gte10_labelled[pair_gte10_labelled["correct"] == 1]
print(
    f"{len(pair_gte10_labelled)} rules with freq >= 10, {len(correct_rules_gte10)} are correct"
)
print(f"Accuracy: {len(correct_rules_gte10) / len(pair_gte10_labelled):.3f}")

pair_1to10_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_pairs_1to10-labelled.xlsx"
)
correct_rules_1to10 = pair_1to10_labelled[pair_1to10_labelled["correct"] == 1]
print(
    f"{len(pair_1to10_labelled)} rules with freq (1, 10), {len(correct_rules_1to10)} are correct"
)
print(f"Accuracy: {len(correct_rules_1to10) / len(pair_1to10_labelled):.3f}")

pair_eq1_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_pairs_eq1-labelled.xlsx"
)
correct_rules_eq1 = pair_eq1_labelled[pair_eq1_labelled["correct"] == 1]
print(
    f"{len(pair_eq1_labelled)} rules with freq = 1, {len(correct_rules_eq1)} are correct"
)
print(f"Accuracy: {len(correct_rules_eq1) / len(pair_eq1_labelled):.3f}")
rules_small = pd.concat(
    [correct_rules_gte10, correct_rules_1to10, correct_rules_eq1]
).sort_values(["commit", "package"], ascending=False)
print(
    f"{len(rules_small)} verified rule in total, {rules_small['package'].nunique()} packages"
)
total_sampled_rules = pd.concat(
    [pair_gte10_labelled, pair_1to10_labelled, pair_eq1_labelled]
).sort_values("commit", ascending=False, ignore_index=True)
total_sampled_rules.to_csv("../benchmark/final/java_labelled_rules.csv", index=False)

338 rules with freq >= 10, 320 are correct
Accuracy: 0.947
102 rules with freq (1, 10), 90 are correct
Accuracy: 0.882
273 rules with freq = 1, 242 are correct
Accuracy: 0.886
652 verified rule in total, 268 packages


In [10]:
commit_pairs_small = (
    rules_small[["package", "old_api", "new_api"]]
    .merge(commit_pairs_full)
    .drop_duplicates()
)
pairs_small = commit_pairs_small[
    ["package", "old_version", "old_api", "new_version", "new_api"]
].drop_duplicates()
print(
    f"{len(commit_pairs_small)} commit pairs, {len(pairs_small)} pairs for {len(rules_small)} correct verified rules"
)
commit_pairs_small.to_json(
    "../benchmark/final/java_commit_pairs_exact.json", orient="records"
)
pairs_small.to_json("../benchmark/final/java_api_pairs_exact.json", orient="records")

13791 commit pairs, 5777 pairs for 652 correct verified rules


In [11]:
new_commit_pairs_full = (
    commit_pairs_full.merge(
        total_sampled_rules[total_sampled_rules["correct"] == 0][
            ["package", "old_api", "new_api"]
        ],
        indicator=True,
        how="left",
    )
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
pairs_full = new_commit_pairs_full[
    ["package", "old_version", "old_api", "new_version", "new_api"]
].drop_duplicates()
rules_all = new_commit_pairs_full[["package", "old_api", "new_api"]].drop_duplicates()
print(
    f"{len(new_commit_pairs_full)} commit pairs, {len(pairs_full)} pairs for all {len(rules_all)} rules after removing incorrect verified rules"
)

new_commit_pairs_full.to_json(
    "../benchmark/final/java_commit_pairs_full.json", orient="records"
)
pairs_full.to_json("../benchmark/final/java_api_pairs_full.json", orient="records")

37793 commit pairs, 26316 pairs for all 15349 rules after removing incorrect verified rules
